# 03 - Categorization & Variable Selection (local port, corrected)

Stage 3: WoE/IV binning of the CAMELS variables, then VIF, correlation and stepwise selection,
ending in a preliminary logit that produces `predicted_good`.

It runs on whatever panel notebook 02 wrote, which is the **prudential conglomerate** panel by
default. Notebook 02 stamps the key it used into an `ID` column, so nothing here needs to know which.
See [`CHANGES_AND_WHY.md`](CHANGES_AND_WHY.md), change N1.

Ported from `course_files/code/03_Categorization_Selection.ipynb`.

**Corrections applied here.** Each one is itemised, justified and measured in
[`CHANGES_AND_WHY.md`](CHANGES_AND_WHY.md) against the instructor's cell number and original
line:

| # | Instructor cell | What changed |
|---|---|---|
| C4 | 18 | `C01` and `A02` excluded from the candidate set. Both are IF.Data capital/RWA series, and `C01` *is* the input to the target rule |
| C5 | 13, 19-25, 32-36, 41-42 | binning, VIF, correlation and stepwise are fit on **`train` only**; previously all of them saw `oot` and `mr` |
| C6 | 10 | split moved inside the COSIF 1.0 regime, because `A01`/`M01` don't exist after 202412 |
| C7 | 25 | guards, so a variable with no usable split or no `Missing` bin gets reported rather than raising `IndexError` |
| C8 | 47 | `mr['predicted_good'] = ...` replaced, since chained assignment on a `query()` slice is a silent no-op under copy-on-write |
| C10 | 18, 22, 23 | the direction check fits both directions and **flips** a variable whose assumed trend the data contradicts, instead of building an exclusion list and ignoring it |

Plus a **compatibility shim** for `optbinning` (next cell), without which nothing here runs at all.

Lecture reference: *Session 5 - Candidate Explanatory Variables*.

### Compatibility shim (must run before `optbinning` is imported)

`optbinning` 0.20 calls scikit-learn's `check_array(..., force_all_finite=...)`. That keyword was
renamed to `ensure_all_finite` in scikit-learn 1.6 and **removed in 1.8**, which is what this machine
has, so every `fit()` dies with `TypeError: check_array() got an unexpected keyword argument
'force_all_finite'`.

`optbinning` 0.21 fixes it but pins `ortools<9.12`, and only ortools 9.15 has Python 3.14 wheels, so
it can't be installed here. Renaming the keyword at the call boundary is the working fix.
`optbinning` does `from sklearn.utils import check_array` at import time, so this cell has to run
first.

In [1]:
import sklearn.utils
import sklearn.utils.validation

_check_array = sklearn.utils.validation.check_array


def check_array(*args, **kwargs):
    if 'force_all_finite' in kwargs:
        kwargs['ensure_all_finite'] = kwargs.pop('force_all_finite')
    return _check_array(*args, **kwargs)


sklearn.utils.check_array = check_array
sklearn.utils.validation.check_array = check_array

import sklearn
print(f"scikit-learn {sklearn.__version__} - check_array shim installed")

scikit-learn 1.8.0 - check_array shim installed


In [2]:
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn import linear_model
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from optbinning import OptimalBinning
from mlxtend.feature_selection import SequentialFeatureSelector

import warnings
warnings.filterwarnings('ignore')

base_dir = pathlib.Path.cwd()
dir_outputs = base_dir / 'Output'

pd.set_option('display.max_rows', 200)
np.random.seed(42)

Load `df_var.parquet`.

Notebook 02 leaves `target` as `NaN` where the BIS proxy can't be evaluated and no explicit default
is on record, and marks the usable rows with `modelling_sample`. We use that distinction two
different ways here, so **every** row is kept:

- **fitting** (binning, selection, coefficients) uses labelled `train` rows only, since an unlabelled
  row teaches the model nothing;
- **scoring** covers every row, because notebooks 04 and 05 have to rate and monitor institutions
  whose 12-month outcome isn't observable yet.

That second point isn't a detail. The target is forward-looking, so the final months of the panel are
unlabelled by construction: you can't yet know whether an institution breaches within the coming
year. Filtering the frame down to labelled rows would silently empty the monitoring sample at exactly
the periods monitoring exists for.

In [3]:
df = pd.read_parquet(dir_outputs / 'df_var.parquet')
ID_COL = 'ID' if 'ID' in df.columns else 'CNPJ'

print(f"df_var          : {df.shape[0]:,} rows ({df['DATA'].min()} to {df['DATA'].max()})")

df = df.query('DATA >= 201703').copy()
labelled = df['modelling_sample']

print(f"kept (>=201703) : {df.shape[0]:,} rows")
print(f"  ...labelled   : {int(labelled.sum()):,} "
      f"({int(df.loc[labelled, 'target'].sum())} bad, {df.loc[labelled, 'target'].mean():.2%})")
print(f"  ...unlabelled : {int((~labelled).sum()):,} (scored, never fitted on)")

df_var          : 14,566 rows (201701 to 202603)
kept (>=201703) : 14,348 rows
  ...labelled   : 13,666 (1249 bad, 9.14%)
  ...unlabelled : 682 (scored, never fitted on)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14348 entries, 218 to 14565
Columns: 312 entries, DATA to PANEL
dtypes: bool(1), float64(295), int64(6), object(10)
memory usage: 34.2+ MB


## Data Partition

Split by **time**, never randomly. The model has to be tested the way it will be used.

**Correction C6.** Session 5 slide 4 puts `train` at 201703-202506 and `oot` at 202507-202512. That
can't work here: `A01` and `M01` are 100% missing from 202501 onward, because `map_cosif_final.xlsx`
has no COSIF 1.5 counterpart for `31500005`, `31800004`, `31900007`, `73000006` or `83000003`. An
out-of-time window in which most explanatory variables don't exist tests nothing.

The split below keeps `train` and `oot` inside the COSIF 1.0 regime and lets `mr` straddle the
accounting break, which is exactly what monitoring is for. The PSI in Session 8 should flag 202501
loudly.

`oot` is calendar 2023. On the conglomerate panel that year carries 207 bad events, comfortably
enough for a stable AUC, and it's the last full year before `A01`, `A02` and `M01` start to
disappear. The instructor's dates and the slide's are both kept as comments so either can be
reproduced.

In [5]:
TRAIN_END = 202212   # instructor's notebook: 202306   |   Session 5 slide 4: 202506
OOT_END = 202312     # instructor's notebook: 202312   |   Session 5 slide 4: 202512


def sample(row):
    if (row['DATA'] >= 201703) & (row['DATA'] <= TRAIN_END):
        return 'train'
    elif (row['DATA'] > TRAIN_END) & (row['DATA'] <= OOT_END):
        return 'oot'
    else:
        return 'mr'


df['sample'] = df[['DATA']].apply(sample, axis=1)

summary = df.groupby('sample').agg(min_DATA=('DATA', 'min'), max_DATA=('DATA', 'max'),
                                   rows=('DATA', 'size'),
                                   labelled=('modelling_sample', 'sum'),
                                   bad=('target', 'sum'))
summary['bad_rate'] = (summary['bad'] / summary['labelled']).map('{:.2%}'.format)
print(summary.to_string())

        min_DATA  max_DATA  rows  labelled    bad bad_rate
sample                                                    
mr        202401    202603  4243      3954  362.0    9.16%
oot       202301    202312  1749      1737  207.0   11.92%
train     201703    202212  8356      7975  680.0    8.53%


In [6]:
train = df.query('sample == "train"')
oot = df.query('sample == "oot"')
mr = df.query('sample == "mr"')

# everything is fitted on the labelled part of train, and only that
train_lab = train[train['modelling_sample']]
y_train = train_lab['target'].astype(int)

print(f"train {len(train):>6,} rows ({len(train_lab):,} labelled, {int(y_train.sum())} bad)")
print(f"oot   {len(oot):>6,} rows ({int(oot['modelling_sample'].sum()):,} labelled)")
print(f"mr    {len(mr):>6,} rows ({int(mr['modelling_sample'].sum()):,} labelled)")

train  8,356 rows (7,975 labelled, 680 bad)
oot    1,749 rows (1,737 labelled)
mr     4,243 rows (3,954 labelled)


**CHECK**: labelled coverage collapses at the panel edge, as it has to.

`target` needs a 12-month forward window, so the most recent periods can't be labelled at all. This
is the honest version of what the instructor's rule concealed by labelling every unobservable row
"bad", and it's the reason scoring and fitting use different row sets.

In [7]:
edge = (df.query('DATA >= 202509')
          .groupby('DATA')['modelling_sample'].agg(labelled='sum', rows='size'))
print(edge.to_string())

        labelled  rows
DATA                  
202509       154   164
202510       158   167
202511       157   166
202512       152   163
202601       151   164
202602       151   163
202603         6   164


## Categorization

### Candidate set and sensible direction

**Correction C4: `C01` and `A02` are excluded.**

- `C01 = BIS`. The target *is* a threshold on that same series (`bis_flag = BIS_linear < minimum`),
  and `ind_default_12m_bis` is forced to 1 whenever today's flag is 1. Using `C01` to predict it is
  circular, not prediction. Session 4 slide 10 already warns that an implicit proxy has to justify
  its "redundancy with explanatory variables".
- `A02 = CRWA_linear / 31000000`. `CRWA` comes from the same IF.Data capital report as the target,
  and IF.Data stops publishing it after 202306, so it's unavailable for `oot` and `mr` anyway.

Both are still constructed in notebook 02; they're simply not offered to the model here. The cell
after selection measures what they were contributing.

`var_asc`: higher value means riskier. `var_desc`: higher value means safer.

In [8]:
var_asc = ['A01']                       # instructor: ['A01', 'A02']
var_desc = ['M01', 'E01', 'L01']        # instructor: ['C01', 'M01', 'E01', 'L01']

EXCLUDED = {'C01': 'target is a threshold on this same BIS series (circular)',
            'A02': 'built from IF.Data CRWA - same source as the target; ends at 202306'}

DIRECTION = {v: 'ascending' for v in var_asc}
DIRECTION.update({v: 'descending' for v in var_desc})
final_var_b = list(DIRECTION)

for v, why in EXCLUDED.items():
    print(f"excluded  {v}: {why}")
print(f"\ncandidates: {final_var_b}")

excluded  C01: target is a threshold on this same BIS series (circular)
excluded  A02: built from IF.Data CRWA - same source as the target; ends at 202306

candidates: ['A01', 'M01', 'E01', 'L01']


A single variable, to see what `OptimalBinning` produces.

**Correction C5** starts here: `ob.fit()` gets `train` only. The instructor's cells 19, 20, 22 and 25
all pass `df[variable].values` and `y = df['target']`, so the whole panel including `oot` and `mr`,
which means the bin edges and WoE values were learned partly from the periods later used to test
them.

In [9]:
variable = 'A01'
ob = OptimalBinning(name=variable, dtype="numerical", solver="cp", prebinning_method="quantile",
                    divergence="iv", monotonic_trend="ascending", max_n_bins=6)
ob.fit(train_lab[variable].values, y_train)   # labelled train only, not df[variable].values
ob.binning_table.build()

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 0.00)",303,0.037994,290,13,0.042904,0.73208,0.015107,0.001847
1,"[0.00, 0.13)",4533,0.568401,4309,224,0.049415,0.583963,0.152570,0.018805
2,"[0.13, 0.18)",303,0.037994,267,36,0.118812,-0.369122,0.006032,0.000750
3,"[0.18, 0.24)",302,0.037868,251,51,0.168874,-0.779224,0.031631,0.003857
4,"[0.24, 0.32)",302,0.037868,229,73,0.241722,-1.229589,0.093402,0.010991
5,"[0.32, inf)",303,0.037994,225,78,0.257426,-1.31346,0.110150,0.012857
6,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
7,Missing,1929,0.241881,1724,205,0.106273,-0.243459,0.015860,0.001978
Totals,,7975,1.000000,7295,680,0.085266,,0.424752,0.051085


### Detection of wrong-direction variables

**Correction C10.** IN03 cell 22 bins every variable under its *assumed* direction and counts the
splits; cell 23 turns the zero-split ones into `split_begin_exclude`, and then nothing ever uses that
list. A variable that yields no splits isn't necessarily uninformative. More often the assumed
direction is simply backwards and the monotonic constraint is fighting the data.

So we fit both directions here, on `train` only, and report them side by side. Where the empirical
direction disagrees with the instructor's assumption the variable gets **flipped**, loudly. Only a
variable that fails in *both* directions is excluded.

On the conglomerate panel two of the four assumptions don't hold, which belongs in the report rather
than being quietly binned around:

- **`L01`** (cash and equivalents / deposits, `11000006` / `41000007`, which isn't a current ratio
  since there's no current-liabilities denominator in it; assumed `descending`, "more liquidity is
  safer"): 0 splits descending, 5 ascending. A low cash/deposits ratio marks a genuinely
  deposit-funded bank, which sits at the safe end of this population. Cash far exceeding deposits
  marks a thin, wholesale-funded or barely-deposit-taking institution, and payment institutions and
  brokers with almost no deposit base dominate this panel.
- **`M01`** (net non-operating result / total assets, `73000006`/`83000003` over `10000007`/
  `20000004`, which isn't a net interest margin since the accounts are *receitas/despesas não
  operacionais* and contain no interest income; assumed `descending`, "more margin is safer"): a
  large non-operating result relative to assets is a distress signal, meaning asset disposals,
  provision write-backs and one-off gains booked to shore up a bad year. That explains rather than
  explains away why the fitted direction comes out ascending.

In [10]:
rows = []
for variable, assumed in DIRECTION.items():
    row = {'var': variable, 'assumed': assumed}
    for trend in ('ascending', 'descending'):
        ob = OptimalBinning(name=variable, dtype="numerical", solver="cp",
                            prebinning_method="quantile", divergence="iv",
                            monotonic_trend=trend, max_n_bins=6)
        ob.fit(train_lab[variable].values, y_train)
        row['splits_' + trend[:3]] = len(ob.splits.tolist())
        row['IV_' + trend[:3]] = float(ob.binning_table.build()['IV'].iloc[-1])
    rows.append(row)

direction_table = pd.DataFrame(rows)
direction_table['empirical'] = np.where(
    direction_table['IV_asc'] > direction_table['IV_des'], 'ascending', 'descending')
direction_table['flipped'] = direction_table['empirical'] != direction_table['assumed']
direction_table['usable_splits'] = np.where(
    direction_table['empirical'] == 'ascending',
    direction_table['splits_asc'], direction_table['splits_des'])

print(direction_table.to_string(index=False))

# adopt the direction the training data actually supports
DIRECTION = dict(zip(direction_table['var'], direction_table['empirical']))
split_begin_exclude = direction_table.query('usable_splits == 0')['var'].to_list()

print()
print("Flipped vs the instructor's assumption:",
      direction_table.query('flipped')['var'].to_list() or 'none')
print("Excluded (no usable split either way) :", split_begin_exclude or 'none')

var    assumed  splits_asc   IV_asc  splits_des   IV_des  empirical  flipped  usable_splits
A01  ascending           5 0.424752           1 0.021685  ascending    False              5
M01 descending           4 0.214094           1 0.057488  ascending     True              4
E01 descending           1 0.144112           4 0.966416 descending    False              4
L01 descending           5 0.198311           0 0.016126  ascending     True              5

Flipped vs the instructor's assumption: ['M01', 'L01']
Excluded (no usable split either way) : none


### Automatic optimal binning

For each variable:

1. bin it **on `train`** to find where the `Missing` bin's event rate sits;
2. substitute missing values with a value inside the closest-behaving real bin (`missing_sub`);
3. re-bin the substituted `train` series and keep that binner;
4. **apply** it to the whole panel, so `oot` and `mr` are transformed by a binner that never saw
   them.

**Correction C7**: the instructor's loop indexes `split[i][0]` and calls `.index(min(...))` with no
check, so a variable with no split or no `Missing` bin raises `IndexError` or `ValueError` instead of
reporting the problem. `M01` produces only one split here, so that's one data change away.

In [11]:
b, bins, woe_binner = {}, {}, {}
iv, missing_sub_table, split, split_final, missing_subs = [], [], {}, {}, {}

for variable, trend in DIRECTION.items():
    if variable in split_begin_exclude:
        print(f"SKIP {variable}: no usable split under a {trend} trend")
        continue

    ob = OptimalBinning(name=variable, dtype="numerical", solver="cp",
                        prebinning_method="quantile", divergence="iv",
                        monotonic_trend=trend, max_n_bins=6)
    ob.fit(train_lab[variable].values, y_train)
    b[variable] = pd.DataFrame(ob.binning_table.build())
    split[variable] = ob.splits.tolist()

    table = ob.binning_table.build().reset_index()
    real_rates = table.query('Bin != "Special" and Bin != "" and Bin != "Missing"')['Event rate'].tolist()
    missing_rate = table.query('Bin == "Missing"')['Event rate'].tolist()

    if not missing_rate or not real_rates or not split[variable]:
        print(f"SKIP {variable}: no Missing bin or no split to substitute into")
        continue

    missing_distance = np.abs(np.subtract(real_rates, missing_rate)).tolist()
    missing_idx = missing_distance.index(min(missing_distance))
    missing_sub = float(np.where(missing_idx == 0,
                                 split[variable][0] - 0.001,
                                 split[variable][missing_idx - 1]))
    missing_subs[variable] = missing_sub
    missing_sub_table.append(pd.DataFrame({variable: [missing_sub]}))

    # refit on train with missing values substituted
    x_train_snull = np.where(pd.isnull(train_lab[variable]), missing_sub,
                             train_lab[variable].values)
    ob_snull = OptimalBinning(name=variable + '_snull', dtype="numerical", solver="cp",
                              prebinning_method="quantile", divergence="iv",
                              monotonic_trend=trend, max_n_bins=6)
    ob_snull.fit(x_train_snull, y_train)
    bins[variable] = pd.DataFrame(ob_snull.binning_table.build())
    split_final[variable] = ob_snull.splits.tolist()
    woe_binner[variable] = ob_snull

    # apply the train-fitted binner to the whole panel
    x_all_snull = np.where(pd.isnull(df[variable]), missing_sub, df[variable].values)
    df[variable + '_woe'] = ob_snull.transform(x_all_snull, metric="woe")

    iv_temp = pd.DataFrame(bins[variable]['IV'][-1:]).rename(columns={'IV': 'IV_' + variable})
    iv.append(iv_temp)

    print(f"OK   {variable}: {len(split_final[variable])} splits, "
          f"IV(train)={float(iv_temp.iloc[0, 0]):.4f}, missing_sub={missing_sub:.6g}")

train = df.query('sample == "train"')
oot = df.query('sample == "oot"')
mr = df.query('sample == "mr"')
train_lab = train[train['modelling_sample']]

OK   A01: 4 splits, IV(train)=0.4193, missing_sub=0.13235
OK   M01: 4 splits, IV(train)=0.2043, missing_sub=0.00028508


OK   E01: 4 splits, IV(train)=0.9664, missing_sub=-0.00168278


OK   L01: 5 splits, IV(train)=0.1912, missing_sub=0.141693


In [12]:
missing_sub_table_export = pd.concat(missing_sub_table, axis=1)
missing_sub_table_export.to_excel(dir_outputs / 'missing_sub_table.xlsx')
missing_sub_table_export

,A01,M01,E01,L01
0,0.13235,0.000285,-0.001683,0.141693


In [13]:
with pd.ExcelWriter(dir_outputs / 'bin.xlsx', engine='xlsxwriter') as writer:
    for var in final_var_b:
        if var in bins:
            bins[var].to_excel(writer, sheet_name=var)
print("Saved bin.xlsx")

Saved bin.xlsx


Information Value per variable, computed on `train`.

Rule of thumb: < 0.02 useless, 0.02-0.1 weak, 0.1-0.3 medium, 0.3-0.5 strong, **> 0.5 suspicious**.
An IV that high usually means the variable is a proxy for the target rather than a predictor of it.

In [14]:
iv_table = pd.concat(iv, axis=1)
iv_table

,IV_A01,IV_M01,IV_E01,IV_L01
Totals,0.41933,0.204255,0.966416,0.191221


**CHECK C4**: what excluding `C01` and `A02` cost, and why they were excluded.

We bin both here **only to report their IV**; neither enters the model. An IV above 0.5 on a variable
drawn from the target's own data source is the signature of leakage, not of a strong predictor.

In [15]:
for variable, trend in [('C01', 'descending'), ('A02', 'ascending')]:
    ob = OptimalBinning(name=variable, dtype="numerical", solver="cp",
                        prebinning_method="quantile", divergence="iv",
                        monotonic_trend=trend, max_n_bins=6)
    ob.fit(train_lab[variable].values, y_train)
    table = ob.binning_table.build()
    print(f"{variable}: IV(train) = {float(table['IV'].iloc[-1]):.4f}   "
          f"(excluded - {EXCLUDED[variable]})")

print("\nCoverage of the excluded variables per sample:")
print(df.groupby('sample')[['C01', 'A02']].apply(lambda g: g.notna().mean())
        .map('{:.1%}'.format).to_string())

C01: IV(train) = 1.1930   (excluded - target is a threshold on this same BIS series (circular))
A02: IV(train) = 0.5480   (excluded - built from IF.Data CRWA - same source as the target; ends at 202306)

Coverage of the excluded variables per sample:
          C01    A02
sample              
mr      32.3%   0.0%
oot     30.5%  33.7%
train   32.3%  78.3%


## Variance Inflation Factor (VIF)

In [16]:
woe_vars = [v + '_woe' for v in final_var_b if v + '_woe' in df.columns]
x_train = train_lab[woe_vars]   # C5: labelled train only (instructor cell 32 used all rows)

vif_data = pd.DataFrame({
    'feature': x_train.columns,
    'VIF': [variance_inflation_factor(x_train.values, i) for i in range(len(x_train.columns))],
})
print(vif_data.to_string(index=False))

feature      VIF
A01_woe 1.160279
M01_woe 1.052808
E01_woe 1.109327
L01_woe 1.066407


## Correlation

In [17]:
corr = x_train.corr().abs()
corr

,A01_woe,M01_woe,E01_woe,L01_woe
A01_woe,1.000000,0.152440,0.219469,0.191724
M01_woe,0.152440,1.000000,0.082583,0.092318
E01_woe,0.219469,0.082583,1.000000,0.088342
L01_woe,0.191724,0.092318,0.088342,1.000000


In [18]:
corr = x_train.corr().abs()
corr[corr == 1] = 0                     # zero out the self-correlation diagonal
high = corr.unstack()
high = high[high > 0.7]
print("Pairs with |corr| > 0.7:")
print(high.to_string() if len(high) else "  none")

Pairs with |corr| > 0.7:
  none


## Variable Selection: Stepwise Regression

**Correction C5** again: the instructor's cells 41 and 42 fit the selector on the whole panel.

In [19]:
sfs_forward = SequentialFeatureSelector(linear_model.LogisticRegression(),
                                        k_features=len(woe_vars), forward=True,
                                        scoring='roc_auc', cv=None)
sfs_forward = sfs_forward.fit(x_train, y_train)
pd.DataFrame(sfs_forward.subsets_).T[['feature_names', 'avg_score']]

,feature_names,avg_score
1,"(E01_woe,)",0.721232
2,"(E01_woe, L01_woe)",0.783956
3,"(A01_woe, E01_woe, L01_woe)",0.802011
4,"(A01_woe, M01_woe, E01_woe, L01_woe)",0.813633


In [20]:
sfs_backward = SequentialFeatureSelector(linear_model.LogisticRegression(),
                                         k_features=1, forward=False,
                                         scoring='roc_auc', cv=None)
sfs_backward = sfs_backward.fit(x_train, y_train)
pd.DataFrame(sfs_backward.subsets_).T[['feature_names', 'avg_score']]

,feature_names,avg_score
4,"(A01_woe, M01_woe, E01_woe, L01_woe)",0.813633
3,"(A01_woe, M01_woe, E01_woe)",0.80713
2,"(A01_woe, E01_woe)",0.78382
1,"(E01_woe,)",0.721232


## Preliminary Model

The model estimates the probability of being a **good** bank, so we invert the target first.
`predicted_good` is what the 1-8 rating scale in notebook 04 will be cut on.

In [21]:
df['y_adj'] = 1 - df['target']
train = df.query('sample == "train"')
oot = df.query('sample == "oot"')
mr = df.query('sample == "mr"')
train_lab = train[train['modelling_sample']]

In [22]:
var_woe = woe_vars

# The instructor's inner random split (cell 45) is kept so the fit stays comparable, even though a
# time-based `oot` already provides the holdout - see CHANGES_AND_WHY.md, open issue O4.
X_train_one, X_test_one, y_train_one, y_test_one = train_test_split(
    train_lab, train_lab['y_adj'].astype(float), test_size=0.30, random_state=42)

logReg = sm.Logit(y_train_one, sm.add_constant(X_train_one[var_woe]))
answer = logReg.fit()
answer.summary()

Optimization terminated successfully.
         Current function value: 0.237953
         Iterations 8


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:                  y_adj   No. Observations:                 5582
Model:                          Logit   Df Residuals:                     5577
Method:                           MLE   Df Model:                            4
Date:                Mon, 31 Aug 2026   Pseudo R-squ.:                  0.2054
Time:                        21:32:23   Log-Likelihood:                -1328.3
converged:                       True   LL-Null:                       -1671.6
Covariance Type:            nonrobust   LLR p-value:                2.566e-147
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.3663      0.056     42.013      0.000       2.256       2.477
A01_woe        0.7255      0.082      8.810      0.000       0.564       0.887
M01_woe        0.9166      0.106      8.668      0.000       0.709       1.124
E01_woe        0.8978      0.047     19.124      0.000       0.806       0.990
L01_woe        0.4964      0.149      3.336      0.001       0.205       0.788
==============================================================================
"""

**Correction C8**: `predicted_good` on every row.

The instructor writes `mr['predicted_good'] = answer.predict(sm.add_constant(df[var_woe]))`
(cell 47). The right-hand side is computed on `df`, not `mr`, and the assignment targets a `query()`
slice. It happens to land the right values through index alignment today, and becomes a silent no-op
under pandas 3 copy-on-write. Predict once on `df`, then rebuild the views.

In [23]:
df['predicted_good'] = answer.predict(sm.add_constant(df[var_woe]))

train = df.query('sample == "train"')
oot = df.query('sample == "oot"')
mr = df.query('sample == "mr"')

print(df.groupby('sample')['predicted_good'].describe()[['count', 'mean', 'min', 'max']].to_string())

         count      mean       min       max
sample                                      
mr      4243.0  0.902649  0.204266  0.992894
oot     1749.0  0.900734  0.133069  0.992894
train   8356.0  0.907967  0.133069  0.992894


**CHECK C5**: a genuine out-of-time test.

Because the binning, the selection and the coefficients now all come from `train` alone, the `oot`
figure below is a real out-of-time result rather than a re-reading of the training data. A visible
train-to-oot drop is expected and healthy. The instructor's version reported a nearly flat curve,
which is what leakage looks like.

In [24]:
for name, part in [('train', train), ('oot', oot), ('mr', mr)]:
    part = part[part['modelling_sample'] & part[var_woe].notna().all(axis=1)]
    if part['y_adj'].nunique() > 1:
        auc = roc_auc_score(part['y_adj'], part['predicted_good'])
        print(f"  AUC ({name:5s}) = {auc:.4f}   n={len(part):,}  "
              f"bad={int((part['y_adj'] == 0).sum())}")
    else:
        print(f"  AUC ({name:5s}) = n/a (one class only)")

  AUC (train) = 0.8141   n=7,975  bad=680
  AUC (oot  ) = 0.8636   n=1,737  bad=207
  AUC (mr   ) = 0.6833   n=3,954  bad=362


**CHECK**: the label no longer follows from missingness.

Under the instructor's rule, `P(target=1 | variable missing)` was 1.0000 for every IF.Data-derived
series. On the corrected label the missing and observed groups have comparable bad rates, which is
what a variable that carries information rather than the answer looks like.

In [25]:
for v in final_var_b + list(EXCLUDED):
    missing = df[v].isna()
    if missing.any() and (~missing).any():
        print(f"  {v}: missing={missing.mean():6.1%}  "
              f"P(target=1 | missing)={df.loc[missing, 'target'].mean():.4f}  "
              f"P(target=1 | observed)={df.loc[~missing, 'target'].mean():.4f}")
    else:
        print(f"  {v}: missing={missing.mean():6.1%}  (no contrast to report)")

  A01: missing= 38.7%  P(target=1 | missing)=0.1058  P(target=1 | observed)=0.0827
  M01: missing= 33.1%  P(target=1 | missing)=0.0959  P(target=1 | observed)=0.0893
  E01: missing=  0.0%  (no contrast to report)
  L01: missing= 12.6%  P(target=1 | missing)=0.1273  P(target=1 | observed)=0.0864
  C01: missing= 67.9%  P(target=1 | missing)=0.0948  P(target=1 | observed)=0.0843
  A02: missing= 50.3%  P(target=1 | missing)=0.1129  P(target=1 | observed)=0.0716


In [26]:
# the institution with the most rows carrying an explicit default flag in its 12-month window
focus = (df[df['ind_default_12m'] == 1].groupby(ID_COL).size().idxmax()
         if (df['ind_default_12m'] == 1).any() else df[ID_COL].iloc[0])
name = df.loc[df[ID_COL] == focus, 'NOME_INSTITUICAO'].iloc[-1]
print(f"worked example: {focus} ({name})")

df[df[ID_COL] == focus][['DATA', ID_COL, 'NOME_INSTITUICAO', 'target',
                         'ind_default_12m', 'predicted_good']].tail(24)

worked example: C0081005 (DACASA FINANCEIRA S/A - SCFI)


,DATA,ID,NOME_INSTITUICAO,target,ind_default_12m,predicted_good
1468,201802,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.906599
1578,201803,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.906599
1688,201804,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.906599
1797,201805,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.906599
1905,201806,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.906599
2013,201807,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.848621
2121,201808,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.906599
2229,201809,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.793769
2338,201810,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.743181
2447,201811,C0081005,DACASA FINANCEIRA S/A - SCFI,1.0,0,0.716104


The most recent monitoring period, worst-rated institutions first.

In [27]:
last_mr = int(mr['DATA'].max())
print(f"Latest monitoring period: {last_mr}")
print(mr.query('DATA == @last_mr')['predicted_good'].describe().to_string())

(mr.query('DATA == @last_mr')[['DATA', ID_COL, 'NOME_INSTITUICAO', 'predicted_good']]
   .sort_values('predicted_good')
   .head(20))

Latest monitoring period: 202603
count    164.000000
mean       0.874759
std        0.127547
min        0.411157
25%        0.903019
50%        0.925755
75%        0.931446
max        0.977808


,DATA,ID,NOME_INSTITUICAO,predicted_good
14485,202603,C0087607,PAGARE IP S.A.,0.411157
14551,202603,C0086354,TRUSTEE DTVM LTDA.,0.411157
14509,202603,C0088840,REAG IP,0.449049
14512,202603,C0085173,COBUCCIO S.A. SCFI,0.449049
14467,202603,C0087061,MERCADO BITCOIN CTVM,0.504677
14413,202603,C0084631,TRINUS CAPITAL DTVM,0.504677
14427,202603,C0083539,BCO AFINZ S.A. - BM,0.504677
14406,202603,C0080587,PLANNER CV S.A.,0.504677
14508,202603,C0085544,FINVEST DTVM,0.504677
14468,202603,C0086361,PINBANK IP,0.504677


In [28]:
df.to_parquet(dir_outputs / 'df_woe.parquet')
print(f"Saved df_woe.parquet - {df.shape[0]:,} rows x {df.shape[1]} columns")

Saved df_woe.parquet - 14,348 rows x 319 columns
